In [3]:
import re
import pandas as pd
import io

In [5]:
with open("support_logs_2025-07-01.log", encoding='utf-8') as f:
    content = f.read()

len(content)

32938

In [6]:
entries = [entry.strip() for entry in content.split("---") if entry.strip()]
entries[0]

'2025-07-01 00:21:00 [INF0] careplus.support.GenericService - TicketID=TCK0701000 SessionID=sess_TCK0701000\nIP=60.130.155.7 | ResponseTime=1269ms | CPU=27.64% | EventType=generic_event | Error=false\nUserAgent="PostmanRuntime/7.32.2"\nMessage=" event for TCK0701000"\nDebug="ℹ️ Logged for monitoring"\nTraceID=None'

In [22]:
# Regex pattern to extract data
log_pattern = re.compile(
    r'(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}) \[(?P<log_level>[A-Za-z0-9_]+)\] '
    r'(?P<component>[^\s]+) - TicketID=(?P<ticket_id>[^\s]+) SessionID=(?P<session_id>[^\s]+)\s*'
    r'IP=(?P<ip>.*?) \| ResponseTime=(?P<response_time>\d+)ms \| CPU=(?P<cpu>[\d.]+)% \| EventType=(?P<event_type>.*?) \| Error=(?P<error>\w+)\s*'
    r'UserAgent=(?P<user_agent>.*?)\s*'
    r'Message=(?P<message>.*?)\s*'
    r'Debug=(?P<debug>.*?)\s*'
    r'TraceID=(?P<trace_id>.*)'
)

# Extract structured data
parsed_entries = []
for entry in entries:
    match = log_pattern.search(entry)
    if match:
        parsed_entries.append(match.groupdict())

parsed_entries[0]


{'timestamp': '2025-07-01 00:21:00',
 'log_level': 'INF0',
 'component': 'careplus.support.GenericService',
 'ticket_id': 'TCK0701000',
 'session_id': 'sess_TCK0701000',
 'ip': '60.130.155.7',
 'response_time': '1269',
 'cpu': '27.64',
 'event_type': 'generic_event',
 'error': 'false',
 'user_agent': '"PostmanRuntime/7.32.2"',
 'message': '" event for TCK0701000"',
 'debug': '"ℹ️ Logged for monitoring"',
 'trace_id': 'None'}

In [23]:
tf = pd.DataFrame(parsed_entries)
tf.head(3)

,timestamp,log_level,component,ticket_id,session_id,ip,response_time,cpu,event_type,error,user_agent,message,debug,trace_id
0,2025-07-01 00:21:00,INF0,careplus.support.GenericService,TCK0701000,sess_TCK0701000,60.130.155.7,1269,27.64,generic_event,false,"""PostmanRuntime/7.32.2""",""" event for TCK0701000""","""ℹ️ Logged for monitoring""",None
1,2025-07-01 00:41:00,INFO,careplus.support.GenericService,TCK0701000,sess_TCK0701000,58.36.189.27,1505,57.24,generic_event,false,"""Mobile-Safari/537.36""",""" event for TCK0701000""","""ℹ️ Logged for monitoring""",None
2,2025-07-01 01:44:00,DEBUG,careplus.support.GenericService,TCK0701001,sess_TCK0701001,181.18.12.170,586,78.43,generic_event,false,"""curl/7.68.0""",""" event for TCK0701001""","""ℹ️ Logged for monitoring""",None


In [24]:
df = tf.drop("trace_id" ,axis=1)
df.head(2)

,timestamp,log_level,component,ticket_id,session_id,ip,response_time,cpu,event_type,error,user_agent,message,debug
0,2025-07-01 00:21:00,INF0,careplus.support.GenericService,TCK0701000,sess_TCK0701000,60.130.155.7,1269,27.64,generic_event,false,"""PostmanRuntime/7.32.2""",""" event for TCK0701000""","""ℹ️ Logged for monitoring"""
1,2025-07-01 00:41:00,INFO,careplus.support.GenericService,TCK0701000,sess_TCK0701000,58.36.189.27,1505,57.24,generic_event,false,"""Mobile-Safari/537.36""",""" event for TCK0701000""","""ℹ️ Logged for monitoring"""


In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 98 entries, 0 to 97
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   timestamp      98 non-null     object
 1   log_level      98 non-null     object
 2   component      98 non-null     object
 3   ticket_id      98 non-null     object
 4   session_id     98 non-null     object
 5   ip             98 non-null     object
 6   response_time  98 non-null     object
 7   cpu            98 non-null     object
 8   event_type     98 non-null     object
 9   error          98 non-null     object
 10  user_agent     98 non-null     object
 11  message        98 non-null     object
 12  debug          98 non-null     object
dtypes: object(13)
memory usage: 10.1+ KB


In [26]:
df = df.astype({
    "response_time": "int",
    "cpu": "float"
})

df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 98 entries, 0 to 97
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   timestamp      98 non-null     object 
 1   log_level      98 non-null     object 
 2   component      98 non-null     object 
 3   ticket_id      98 non-null     object 
 4   session_id     98 non-null     object 
 5   ip             98 non-null     object 
 6   response_time  98 non-null     int64  
 7   cpu            98 non-null     float64
 8   event_type     98 non-null     object 
 9   error          98 non-null     object 
 10  user_agent     98 non-null     object 
 11  message        98 non-null     object 
 12  debug          98 non-null     object 
dtypes: float64(1), int64(1), object(11)
memory usage: 10.1+ KB


In [27]:
import pandas as pd

# Convert 'error' column (string "true"/"false") into boolean True/False
df['error'] = df['error'].str.lower().map({'true': True, 'false': False})

# Convert 'timestamp' column into datetime format
df['timestamp'] = pd.to_datetime(
    df['timestamp'],
    format='%Y-%m-%d %H:%M:%S',
    errors='coerce'
).astype('datetime64[ns]')


In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 98 entries, 0 to 97
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   timestamp      98 non-null     datetime64[ns]
 1   log_level      98 non-null     object        
 2   component      98 non-null     object        
 3   ticket_id      98 non-null     object        
 4   session_id     98 non-null     object        
 5   ip             98 non-null     object        
 6   response_time  98 non-null     int64         
 7   cpu            98 non-null     float64       
 8   event_type     98 non-null     object        
 9   error          98 non-null     bool          
 10  user_agent     98 non-null     object        
 11  message        98 non-null     object        
 12  debug          98 non-null     object        
dtypes: bool(1), datetime64[ns](1), float64(1), int64(1), object(9)
memory usage: 9.4+ KB


In [29]:
df.describe()

,timestamp,response_time,cpu
count,98,98.000000,98.000000
mean,2025-07-01 08:24:49.591836416,1006.500000,54.821633
min,2025-07-01 00:21:00,126.000000,13.350000
25%,2025-07-01 05:39:15,653.250000,34.710000
50%,2025-07-01 09:15:00,1089.000000,59.455000
75%,2025-07-01 11:33:00,1327.750000,73.242500
max,2025-07-01 14:10:00,1792.000000,89.970000
std,NaN,447.808944,22.185337


In [30]:
df["log_level"].value_counts()

log_level
INFO       37
DEBUG      32
INF0       13
DEBG       10
warnING     3
WARNING     3
Name: count, dtype: int64

In [31]:
fix_log_level = {'INF0': 'INFO', 'DEBG': 'DEBUG', 'warnING': 'WARNING', 'EROR': 'ERROR'}
df['log_level'] = df['log_level'].replace(fix_log_level)

df.log_level.value_counts()

log_level
INFO       50
DEBUG      42
WARNING     6
Name: count, dtype: int64

In [32]:
df = df.drop_duplicates()

In [33]:
df.shape

(89, 13)